# FT-CUR — Probe de Escassez nos Datasets Geométricos

Responde a uma ressalva metodológica específica do Apêndice C.

**O problema:** no LSSVM-Nyström, o $k$-means só superou o `random` nos datasets
geométricos num regime de **escassez FIXA** ($m/n = 10\%$) — no ponto de operação
era empate. Já o FT-CUR foi testado apenas no ponto de operação (`m_ratio` tunado
0,10/0,20). Comparar "FT-CUR tunado" com "LSSVM fixo-10%" é maçã com laranja.

**O que este notebook faz:** repete a ablação de escassez ($m$ FIXO em 10% e 5%)
para o FT-CUR, nos 3 sintéticos geométricos (TWS/espiral, TWC/tabuleiro,
TWM/luas), com os 4 seletores e 30 sementes.

**O que decide:**
- Se o `kmeans` **empatar** o `random` → a afirmação do apêndice fica rigorosa:
  *nem no regime escasso a seleção resgata o FT-CUR* (gargalo = viés indutivo).
- Se o `kmeans` **ajudar** → o FT-CUR se comporta como o LSSVM, e a seção é reescrita.

**Grade:** `n_layers × n_heads` (6 combos) — o `m_ratio` sai da grade e fica FIXO,
que é justamente o ponto (isola o regime escasso). Demais hiperparâmetros idênticos
aos do Tier 1.

**Custo estimado (T4):** ~3h no total (720 runs × ~15 s). Resumível.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2** (ou P100).


In [ ]:
# Célula 1 — GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — Settings > Accelerator > GPU')

In [ ]:
# Célula 2 — Clonar repo (branch da ablação)
import os, subprocess
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
BRANCH  = 'ftcur-selection-ablation'
PROJECT = '/kaggle/working/sparse-lssvm-transformers-study'
if os.path.exists(PROJECT):
    subprocess.run(['git','-C',PROJECT,'fetch','origin'], check=True)
    subprocess.run(['git','-C',PROJECT,'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',PROJECT,'pull','--rebase','origin',BRANCH], check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,GIT_URL,PROJECT], check=True)
os.chdir(PROJECT)
!git log --oneline -2
print('Dir:', os.getcwd())

In [ ]:
# Célula 3 — Dependências
!pip install -q entmax einops scipy
import torch, sklearn, numpy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__}')

In [ ]:
# Célula 4 — Datasets (os 3 geométricos são sintéticos, gerados pelo loader)
import sys; sys.path.insert(0, '.')
from src.data.loaders import DatasetLoader
for ds in ['TWS','TWC','TWM']:
    X, y, _ = DatasetLoader.load(ds)
    print(f'  {ds:<5} N={len(y):>5}  p={X.shape[1]}  (classes: {sorted(set(y))})')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Célula 5 — SMOKE TEST (rápido: 4 seletores × TWS × seed 0, m=10%)
# Valida grid=6, m fixo, esparsidade 0,90 e que os 4 rodam sem erro.
# ═══════════════════════════════════════════════════════════════════════
import json, time
V = ('FTTransformerCURColnorm FTTransformerCURRandom '
     'FTTransformerCURKmeans FTTransformerCUROpposite')
t0 = time.time()
!python -u scripts/run_nystrom_scarce_ablation.py --variants {V} \
    --datasets TWS --seeds 0 --m-ratio 0.10 \
    --output /kaggle/working/smoke.json 2>&1 | tail -8
print(f'\n--- smoke: {time.time()-t0:.0f}s ---')
for r in json.load(open('/kaggle/working/smoke.json')):
    s = r['status']
    print(f"{r['variant']:<26} {s:<6}",
          (f"F1={r['test_f1_macro']:.4f} m={r.get('n_support_vectors')} "
           f"spars={r.get('sparsity_ratio',0):.3f} grid={r['grid_size']} "
           f"m_fix={r['m_ratio_fixed']} fit={r['fit_time_s']:.0f}s")
          if s == 'ok' else r.get('error','')[:80])

In [ ]:
# Célula 6 — Config
V_LIST = ['FTTransformerCURColnorm','FTTransformerCURRandom',
          'FTTransformerCURKmeans','FTTransformerCUROpposite']
V = ' '.join(V_LIST)
GEO = 'TWS TWC TWM'
N_SEEDS = 30
SEEDS = ' '.join(map(str, range(N_SEEDS)))
F10 = 'results/ftcur_scarce_m10_geo.json'
F05 = 'results/ftcur_scarce_m05_geo.json'
print(f'Plano por regime: {len(V_LIST)} seletores × 3 datasets × {N_SEEDS} seeds '
      f'= {len(V_LIST)*3*N_SEEDS} runs')
print('Total (2 regimes):', len(V_LIST)*3*N_SEEDS*2, 'runs')

In [ ]:
# Célula 7 — m = 10%  (o regime CRÍTICO: é onde o k-means ajudou o LSSVM)
# Log AO VIVO (sem pipe pra tail — dá pra acompanhar cada run terminando).
!python -u scripts/run_nystrom_scarce_ablation.py --variants {V} \
    --datasets {GEO} --seeds {SEEDS} --m-ratio 0.10 \
    --output {F10} --log-level INFO 2>&1 | tee /kaggle/working/run_m10.log
import shutil; shutil.copy(F10, '/kaggle/working/')
print('\n>>> m=10% salvo em /kaggle/working/ <<<')

In [ ]:
# Célula 8 — m = 5%  (regime complementar; no LSSVM o k-means PIORAVA aqui)
!python -u scripts/run_nystrom_scarce_ablation.py --variants {V} \
    --datasets {GEO} --seeds {SEEDS} --m-ratio 0.05 \
    --output {F05} --log-level INFO 2>&1 | tee /kaggle/working/run_m05.log
import shutil; shutil.copy(F05, '/kaggle/working/')
print('\n>>> m=5% salvo em /kaggle/working/ <<<')

In [ ]:
# Célula 9 — ANÁLISE: o k-means resgata o FT-CUR na geometria escassa?
import json, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon, friedmanchisquare
SELS = ['Colnorm','Random','Kmeans','Opposite']
for label, f in [('m = 10%', F10), ('m = 5%', F05)]:
    p = Path(f)
    if not p.exists():
        print(f'\n=== {label}: (sem dados) ==='); continue
    d = [r for r in json.loads(p.read_text()) if r['status']=='ok']
    S = {v: {(r['dataset'],r['seed']): r['test_f1_macro']
             for r in d if r['variant']=='FTTransformerCUR'+v} for v in SELS}
    print(f'\n=== {label} — F1-macro (geométricos) ===')
    print(f"{'dataset':7s} {'colnorm':>8s} {'random':>8s} {'kmeans':>8s} {'opposite':>9s} "
          f"{'km-rnd':>8s} {'p(km)':>7s}  Friedman")
    for ds in ['TWS','TWC','TWM']:
        ks = sorted(set.intersection(*[{k[1] for k in S[v] if k[0]==ds} for v in SELS]))
        if len(ks) < 5:
            print(f'{ds:7s}  (só {len(ks)} pares)'); continue
        vals = {v: [S[v][(ds,s)] for s in ks] for v in SELS}
        pk = wilcoxon(vals['Kmeans'], vals['Random'])[1]
        fp = friedmanchisquare(*[vals[v] for v in SELS])[1]
        dk = st.mean(vals['Kmeans']) - st.mean(vals['Random'])
        star = '*' if pk < 0.05 else ' '
        print(f"{ds:7s} {st.mean(vals['Colnorm']):8.4f} {st.mean(vals['Random']):8.4f} "
              f"{st.mean(vals['Kmeans']):8.4f} {st.mean(vals['Opposite']):9.4f} "
              f"{dk:+8.4f} {pk:7.3f}{star} {fp:7.3f}  (n={len(ks)})")
print('\n>>> COLE ESTA SAÍDA DE VOLTA NO CHAT <<<')
print('Se kmeans NÃO superar random (p>0,05): a afirmação do Apêndice C fica rigorosa.')
print('Se superar: o FT-CUR se comporta como o LSSVM e a seção será reescrita.')